# IBM Python Project for Data Science — Submission Notebook

このノートブックは Coursera のハンズオンラボ（`Extracting and Visualizing Stock Data`）の提出用テンプレートです。

含まれるセクション:
1. Standard version (yfinance + web scraping)
2. Alternative API version (pandas-datareader / Alpha Vantage案)
3. Matplotlib-only plotting version

> **注意**: この実行環境ではインターネット接続が無効なため、実際のデータ取得セルはローカル環境で実行してください.

## Imports and setup
必要なライブラリをインポートします。実行前に `yfinance`, `beautifulsoup4`, `plotly`, `pandas_datareader` (代替), `alpha_vantage` (代替) をインストールしてください。

In [ ]:
# Standard imports
import pandas as pd
import numpy as np
import yfinance as yf
import requests
from bs4 import BeautifulSoup
import plotly.graph_objects as go
import matplotlib.pyplot as plt
from datetime import datetime

# Utility
pd.options.display.max_columns = 50


## Q1: yfinance を使って株価データを取得
以下は Tesla の株価を取得する例です。`period='max'` の代わりに必要な期間を指定して下さい。

In [ ]:
# Q1: Tesla stock data with yfinance
tesla = yf.Ticker('TSLA')
tesla_data = tesla.history(period='max')
tesla_data = tesla_data.reset_index()
display(tesla_data.head())


## Q2: Webスクレイピングで Tesla の Revenue を取得（例）
Yahoo Finance のページ構成は変わることがあるため、スクレイピングは脆弱です。実行環境によってはブロックされることがあります。

In [ ]:
# Q2: Tesla revenue scraping (example - may need adjustments)
url = 'https://finance.yahoo.com/quote/TSLA/financials?p=TSLA'
resp = requests.get(url)
soup = BeautifulSoup(resp.text, 'html.parser')

# 探索用: ページのどの部分にテーブルがあるか見る
# print(soup.prettify()[:1000])

# Yahoo Finance の財務行要素を探す（サイト構成により変更が必要）
rows = soup.find_all('div', {'data-test': 'fin-row'})
revenue_values = []
for row in rows:
    if 'Total Revenue' in row.get_text():
        cols = row.find_all('div', recursive=False)
        for c in cols[1:]:
            revenue_values.append(c.get_text())

tesla_revenue = pd.DataFrame({
    'Date': [str(x) for x in range(2024, 2024 - len(revenue_values), -1)],
    'Revenue': revenue_values
})
display(tesla_revenue.head())


## Q3: GameStop の株価取得
同様に GME を取得します。

In [ ]:
# Q3: GameStop (GME)
gme = yf.Ticker('GME')
gme_data = gme.history(period='max').reset_index()
display(gme_data.head())


## Q4: GameStop の Revenue をスクレイピング（例）

In [ ]:
# Q4: GME revenue scraping (example - may need adjustments)
url_gme = 'https://finance.yahoo.com/quote/GME/financials?p=GME'
resp = requests.get(url_gme)
soup = BeautifulSoup(resp.text, 'html.parser')

rows = soup.find_all('div', {'data-test': 'fin-row'})
revenue_values_gme = []
for row in rows:
    if 'Total Revenue' in row.get_text():
        cols = row.find_all('div', recursive=False)
        for c in cols[1:]:
            revenue_values_gme.append(c.get_text())

gme_revenue = pd.DataFrame({
    'Date': [str(x) for x in range(2024, 2024 - len(revenue_values_gme), -1)],
    'Revenue': revenue_values_gme
})
display(gme_revenue.head())


## Q5: Plot 関数（Plotly）
Plotly を使用してインタラクティブなグラフを描画する関数を用意します。

In [ ]:
def make_stock_graph(stock_data, title='Stock Price'):
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=stock_data['Date'], y=stock_data['Close'], mode='lines', name='Close'))
    fig.update_layout(title=title, xaxis_title='Date', yaxis_title='Price (USD)')
    fig.show()

def make_stock_and_revenue_graph(stock_data, revenue_data, title='Stock & Revenue'):
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=stock_data['Date'], y=stock_data['Close'], mode='lines', name='Stock Price'))

    fig.add_trace(go.Bar(x=revenue_data['Date'], y=revenue_data['Revenue'], name='Revenue', yaxis='y2', opacity=0.6))

    fig.update_layout(
        title=title,
        xaxis=dict(title='Date / Year'),
        yaxis=dict(title='Stock Price (USD)'),
        yaxis2=dict(title='Revenue (USD)', overlaying='y', side='right')
    )
    fig.show()


## Q6: グラフ描画の呼び出し例
下のセルは、データが用意できたら実行してください。

In [ ]:
# Example usage (実データがある場合に実行)
# make_stock_graph(tesla_data, 'Tesla Stock Price')
# make_stock_and_revenue_graph(tesla_data, tesla_revenue, 'Tesla: Price & Revenue')

# make_stock_graph(gme_data, 'GME Stock Price')
# make_stock_and_revenue_graph(gme_data, gme_revenue, 'GME: Price & Revenue')


## Alternative: Web scraping がブロックされた場合の代替案
- `pandas_datareader` を使う
- Alpha Vantage の API を使う（APIキーが必要）

下は Alpha Vantage を使うサンプル（実行には `alpha_vantage` ライブラリのインストールと API KEY が必要です）。

In [ ]:
# Alpha Vantage sample (placeholder) - requires API key and library
# from alpha_vantage.timeseries import TimeSeries
# api_key = 'YOUR_ALPHA_VANTAGE_KEY'
# ts = TimeSeries(key=api_key, output_format='pandas')
# data, meta = ts.get_daily(symbol='TSLA', outputsize='full')
# data = data.reset_index().rename(columns={'date':'Date', '4. close':'Close'})
# display(data.head())

# pandas_datareader sample (placeholder)
# from pandas_datareader import data as pdr
# import yfinance as yf
# yf.pdr_override()
# df = pdr.get_data_yahoo('TSLA', start='2010-01-01', end=datetime.today().strftime('%Y-%m-%d'))
# df = df.reset_index()
# display(df.head())


## Matplotlib-only: Plotly が使えない環境向け
Matplotlib で株価と収益を可視化する例を示します。

In [ ]:
def plot_with_matplotlib(stock_data, revenue_data=None, title='Stock & Revenue (Matplotlib)'):
    plt.figure(figsize=(12,6))
    plt.plot(stock_data['Date'], stock_data['Close'], label='Stock Price')
    if revenue_data is not None:
        ax2 = plt.twinx()
        rev = revenue_data['Revenue'].astype(str).str.replace(',', '').astype(float)
        ax2.plot(revenue_data['Date'], rev, marker='o', label='Revenue')
        ax2.set_ylabel('Revenue (USD)')
    plt.title(title)
    plt.xlabel('Date')
    plt.ylabel('Price (USD)')
    plt.legend(loc='upper left')
    plt.tight_layout()
    plt.show()


## 実行上の注意と提出時チェックリスト
- ノートブック内のスクレイピングは実行環境に依存します。実行できない場合は代替 API を利用してください。
- 必要なライブラリを `pip install` してください: `yfinance`, `beautifulsoup4`, `plotly`, `pandas_datareader`, `alpha_vantage`（代替）
- 提出前にセルを一度最後まで実行してエラーがないか確認してください。

---

作成者: ChatGPT
生成日時: 2025-11-15 01:11:51 UTC
